# VaR Backtesting and Model Validation

This notebook demonstrates how to backtest VaR models using:
- Kupiec POF Test
- Christoffersen Test
- Traffic Light Approach

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from market_risk_hub.data.market_data import MarketDataFetcher
from market_risk_hub.risk_engines.var import VaRCalculator
from market_risk_hub.backtesting.var_backtest import VaRBacktest
from market_risk_hub.utils.visualization import RiskVisualizer

## 1. Fetch Historical Data

In [ ]:
fetcher = MarketDataFetcher()
tickers = ['SPY']  # S&P 500 ETF

data = fetcher.get_market_data(tickers, period='5y')
returns = data['returns']['SPY']

print(f"Data loaded: {len(returns)} observations")
print(f"Date range: {returns.index[0]} to {returns.index[-1]}")

## 2. Calculate Rolling VaR Estimates

In [ ]:
# Initialize VaR calculator
var_calc = VaRCalculator(confidence_level=0.95)

# Rolling window parameters
window = 252  # 1 year

# Calculate rolling VaR
rolling_var = []
actual_returns = []

for i in range(window, len(returns)):
    # Historical returns for estimation
    historical_returns = returns.iloc[i-window:i]
    
    # Calculate VaR
    var = var_calc.historical_var(historical_returns)
    rolling_var.append(var)
    
    # Actual return for next period
    actual_returns.append(returns.iloc[i])

# Convert to Series
rolling_var = pd.Series(rolling_var, index=returns.index[window:])
actual_returns = pd.Series(actual_returns, index=returns.index[window:])

print(f"Backtesting period: {len(rolling_var)} days")

## 3. Run Backtests

In [ ]:
# Initialize backtest
backtest = VaRBacktest(confidence_level=0.95)

# Run comprehensive backtest
results = backtest.comprehensive_backtest(actual_returns, rolling_var)

print("Backtest Results")
print("=" * 70)

## 4. Kupiec POF Test Results

In [ ]:
kupiec = results['kupiec']

print("Kupiec Proportion of Failures (POF) Test")
print("=" * 70)
print(f"Test Type: {kupiec['test']}")
print(f"Total Observations: {kupiec['total_observations']}")
print(f"Number of Exceptions: {kupiec['exceptions']}")
print(f"Exception Rate: {kupiec['exception_rate']:.2%}")
print(f"Expected Rate: {kupiec['expected_rate']:.2%}")
print(f"LR Statistic: {kupiec['lr_statistic']:.4f}")
print(f"P-Value: {kupiec['p_value']:.4f}")
print(f"Reject Null: {kupiec['reject_null']}")
print(f"Interpretation: {kupiec['interpretation']}")

## 5. Christoffersen Test Results

In [ ]:
cc = results['christoffersen']

print("Christoffersen Conditional Coverage Test")
print("=" * 70)
print(f"Test Type: {cc['test']}")
print(f"Number of Exceptions: {cc['exceptions']}")
print(f"LR Unconditional: {cc['lr_unconditional']:.4f}")
print(f"LR Independence: {cc['lr_independence']:.4f}")
print(f"LR CC Statistic: {cc['lr_cc_statistic']:.4f}")
print(f"P-Value: {cc['p_value']:.4f}")
print(f"Reject Null: {cc['reject_null']}")
print(f"Interpretation: {cc['interpretation']}")

## 6. Traffic Light Test Results

In [ ]:
tl = results['traffic_light']

print("Traffic Light Test (Basel Approach)")
print("=" * 70)
print(f"Test Type: {tl['test']}")
print(f"Total Observations: {tl['total_observations']}")
print(f"Number of Exceptions: {tl['exceptions']}")
print(f"Green Threshold: {tl['green_threshold']}")
print(f"Yellow Threshold: {tl['yellow_threshold']}")
print(f"Zone: {tl['zone']}")
print(f"Action Required: {tl['action']}")

## 7. Visualize Backtest Results

In [ ]:
# Prepare data for visualization
backtest_df = backtest.plot_backtest_results(actual_returns, rolling_var)

# Create visualization
fig = RiskVisualizer.plot_backtest_results(backtest_df)
fig.show()

## 8. Exception Analysis

In [ ]:
# Identify exceptions
exceptions = backtest.calculate_exceptions(actual_returns, rolling_var)
exception_dates = exceptions[exceptions == 1].index

print(f"VaR Breaches ({len(exception_dates)} total):")
print("=" * 70)

for date in exception_dates:
    loss = -actual_returns[date]
    var_est = rolling_var[date]
    excess = loss - var_est
    print(f"{date.date()}: Loss={loss:.4f}, VaR={var_est:.4f}, Excess={excess:.4f}")

## 9. Compare Different VaR Methods

In [ ]:
# Compare Historical vs Parametric VaR
rolling_var_parametric = []

for i in range(window, len(returns)):
    historical_returns = returns.iloc[i-window:i]
    var = var_calc.parametric_var(historical_returns)
    rolling_var_parametric.append(var)

rolling_var_parametric = pd.Series(rolling_var_parametric, index=returns.index[window:])

# Backtest parametric VaR
results_parametric = backtest.comprehensive_backtest(actual_returns, rolling_var_parametric)

print("Comparison: Historical vs Parametric VaR")
print("=" * 70)
print(f"Historical VaR Exceptions: {results['kupiec']['exceptions']}")
print(f"Parametric VaR Exceptions: {results_parametric['kupiec']['exceptions']}")
print(f"\nHistorical VaR: {results['kupiec']['interpretation']}")
print(f"Parametric VaR: {results_parametric['kupiec']['interpretation']}")